# Undersample once, train several models

In the previous notebook the undersampler lived inside the pipeline, so it re-ran on every fold and every candidate during a search. That is correct, but wasteful when we compare several models on the same undersampled data, especially with a slow cleaning method like the Neighbourhood Cleaning Rule (NCR).

Here we take the more efficient route: we apply NCR **once** to build the cross-validation folds and the full training set, and then reuse those pre-undersampled sets to tune and train both XGBoost and a Random Forest. NCR runs a fixed handful of times, and the result is shared across every candidate and both models.

The evaluation remains accurate because training folds are undersampled, while validation folds keep the original class distribution.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="Could not infer format")

In [2]:
import numpy as np
import pandas as pd

from feature_engine.encoding import OrdinalEncoder

from sklearn.base import clone
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterSampler
from sklearn.preprocessing import MinMaxScaler

from imblearn.under_sampling import NeighbourhoodCleaningRule

from xgboost import XGBClassifiercfS

## Load the data

We use the Bank Marketing dataset again, where the minority class is the clients who subscribed to a term deposit.

In [3]:
data = fetch_openml(name="bank-marketing", version=1, as_frame=True, parser="auto")

X = OrdinalEncoder(encoding_method="arbitrary").fit_transform(data.data)
y = (data.target == "2").astype(int)  # 1 = subscribed, 0 = did not subscribe

print(f"Observations: {X.shape[0]}, features: {X.shape[1]}")
y.value_counts(normalize=True)

Observations: 45211, features: 16


Class
0    0.883015
1    0.116985
Name: proportion, dtype: float64

## Split into train and test

We hold out a test set at the original class distribution, and convert the training data to arrays for the folding step.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X.to_numpy(), y.to_numpy(), test_size=0.3, random_state=0, stratify=y
)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

Train: 31647 rows | Test: 13564 rows


## Configure the Neighbourhood Cleaning Rule

We set NCR with the same parameters used in the research code (`configs/undersamplers.py`). NCR relies on nearest-neighbor searches, so we scale the features to guide the neighbor search, then keep the selected rows in their original scale. Tree models do not need scaled inputs, so training happens on the raw values. `sample_indices_` tells us which rows NCR kept.

In [5]:
ncr = NeighbourhoodCleaningRule(
    sampling_strategy="auto",
    n_neighbors=3,
    threshold_cleaning=0.5,
)


def undersample_data(undersampler, X, y):
    """Undersample using scaled features to guide the neighbour search,
    return the kept rows in their original scale."""
    X_scaled = MinMaxScaler().fit_transform(X)
    undersampler.fit_resample(X_scaled, y)
    idx = undersampler.sample_indices_
    return X[idx], y[idx]

## Undersample once

We now apply NCR a single time to each training fold to build the tuning folds, and once more to the full training set for the final refit. The validation folds are left untouched, so they keep the original class distribution.

This is the whole point of the notebook: NCR runs only `n_splits + 1` times in total (six here), and these pre-undersampled sets are reused for every hyperparameter candidate and for both models. In the pipeline approach, NCR would instead run once per fold, per candidate, per model.

In [6]:
def make_us_folds(undersampler, X, y, n_splits=3, seed=10):
    """Undersample each training fold once. Validation folds stay at the
    original class distribution. Returns a list of (Xu, yu, X_val, y_val)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for train_idx, val_idx in skf.split(X, y):
        Xu, yu = undersample_data(undersampler, X[train_idx], y[train_idx])
        folds.append((Xu, yu, X[val_idx], y[val_idx]))
    return folds


# Pre-undersampled folds for tuning.
folds = make_us_folds(ncr, X_train, y_train, n_splits=3, seed=10)

# Full undersampled training set for the final refit.
X_res, y_res = undersample_data(ncr, X_train, y_train)

removed_pct = round((1 - len(X_res) / len(X_train)) * 100, 1)
print(f"Full training set: {len(X_train)} -> {len(X_res)} rows ({removed_pct}% removed)")
print(f"NCR runs used: {len(folds)} folds + 1 full set = {len(folds) + 1}")

Full training set: 31647 -> 26402 rows (16.6% removed)
NCR runs used: 3 folds + 1 full set = 4


## A tuning routine that reuses the pre-undersampled folds

The `tune` function below is a plain random search run sequentially: it samples `n_iter` hyperparameter combinations, trains each one on the pre-undersampled training folds, and scores it on the untouched validation folds, keeping the best. Because the folds were undersampled once upfront, the undersampler is never called inside the loop.

This is equivalent to scikit-learn's `RandomizedSearchCV`. We write it by hand only because the training folds are undersampled while the validation folds keep the original class distribution, which a standard cross-validation splitter cannot express.

In [7]:
def tune(model, param_grid, folds, n_iter=20, seed=42):
    """Random search over pre-undersampled folds. Returns the best params
    and the mean cross-validation ROC-AUC."""
    candidates = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=seed))
    best_score, best_params = -np.inf, None
    for params in candidates:
        fold_scores = []
        for Xu, yu, X_val, y_val in folds:
            clf = clone(model).set_params(**params)
            clf.fit(Xu, yu)
            proba = clf.predict_proba(X_val)[:, 1]
            fold_scores.append(roc_auc_score(y_val, proba))
        mean_score = np.mean(fold_scores)
        std_score = np.std(fold_scores)
        if mean_score > best_score:
            best_score, best_std, best_params = mean_score, std_score, params
    return best_params, best_score, best_std

## Tune and train XGBoost

We tune XGBoost on the pre-undersampled folds, then refit the best configuration on the full undersampled training set.

In [8]:
xgb = XGBClassifier(random_state=10, n_jobs=-1, eval_metric="logloss")

xgb_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [2, 3, 4, 6],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}

xgb_best_params, xgb_cv_score, xgb_cv_std = tune(xgb, xgb_param_grid, folds, n_iter=20)

xgb_final = clone(xgb).set_params(**xgb_best_params)
xgb_final.fit(X_res, y_res)

print(f"XGBoost best CV ROC-AUC: {xgb_cv_score:.4f} +/- {xgb_cv_std:.4f}")
print(f"XGBoost best params: {xgb_best_params}")

XGBoost best CV ROC-AUC: 0.9321 +/- 0.0014
XGBoost best params: {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.7}


## Tune and train the Random Forest

We reuse the **same** pre-undersampled folds and full training set. No further undersampling is needed.

In [9]:
rf = RandomForestClassifier(random_state=10, n_jobs=-1)

rf_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [4, 6, 8, None],
    "max_features": ["sqrt", 0.5],
    "min_samples_leaf": [1, 5, 20],
}

rf_best_params, rf_cv_score, rf_cv_std = tune(rf, rf_param_grid, folds, n_iter=20)

rf_final = clone(rf).set_params(**rf_best_params)
rf_final.fit(X_res, y_res)

print(f"Random Forest best CV ROC-AUC: {rf_cv_score:.4f} +/- {rf_cv_std:.4f}")
print(f"Random Forest best params: {rf_best_params}")

Random Forest best CV ROC-AUC: 0.9301 +/- 0.0011
Random Forest best params: {'n_estimators': 300, 'min_samples_leaf': 5, 'max_features': 0.5, 'max_depth': None}


## Compare the two models on the held-out test set

Both models were trained on the same NCR-undersampled data. We now score them on the test set, which was never undersampled, using ROC-AUC and PR-AUC.

In [11]:
results = []
for name, model in [("XGBoost", xgb_final), ("Random Forest", rf_final)]:
    proba = model.predict_proba(X_test)[:, 1]
    results.append(
        {
            "model": name,
            "test_roc_auc": roc_auc_score(y_test, proba),
            "test_pr_auc": average_precision_score(y_test, proba),
        }
    )

print(pd.DataFrame(results).set_index("model").round(4))

               test_roc_auc  test_pr_auc
model                                   
XGBoost              0.9332       0.6211
Random Forest        0.9310       0.6062


## Takeaway

By undersampling once and reusing the result, NCR ran only six times in total, no matter how many hyperparameter candidates we tried or how many models we compared. The pipeline approach would have re-run NCR on every fold, for every candidate, for each model. Undersampling once upfront removes that redundant work, which matters most for the slower cleaning methods, and it lets us tune and compare XGBoost and the Random Forest on identical undersampled data.